In [1]:
import sys
sys.path.insert(0, "/Users/girishkrishna/Documents/MSSE_Du/277B Machine Learning/Project_Code/Evidence-Aware-Validation-of-AI-Generated-Drug-Candidates-Using-PPI-Networks")

from Compund_Matching_Engine import (
    parse_smiles,
    FingerprintEncoder,
    DrugLikenessFilter,
    ScaffoldExtractor,
    StructureVisualiser,
    CompoundLoader,
    SimilarityScorer,
    MatchResult,
    FilterResult,
)

print("All modules imported successfully!")

AI-generated molecule: OK
Broken SMILES: ERROR: invalid SMILES string


[00:58:17] SMILES Parse Error: syntax error while parsing: CC(C)c1ccc(INVALID!!!)cc1
[00:58:17] SMILES Parse Error: Failed parsing SMILES 'CC(C)c1ccc(INVALID!!!)cc1' for input: 'CC(C)c1ccc(INVALID!!!)cc1'


<class 'rdkit.DataStructs.cDataStructs.ExplicitBitVect'>
2048
Bits ON: 14
Numpy array shape: (2048,)
Numpy array type: int8
All modules imported successfully!


In [4]:
# --- A2: Parse SMILES ---
test_smiles = "CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O"  # Atorvastatin
mol, status = parse_smiles(test_smiles)
print(f"Parse status: {status}")

# --- A3: Fingerprint encoding ---
encoder = FingerprintEncoder()
fp = encoder.encode(mol)
print(f"Fingerprint bits: {fp.GetNumBits()}, on-bits: {fp.GetNumOnBits()}")

# --- A6/A7: Drug-likeness filter ---
dl_filter = DrugLikenessFilter()
filter_result = dl_filter.filter(mol)
print(f"Drug-likeness: clean={filter_result.is_clean}, violations={filter_result.violations}")

# --- A8: Scaffold extraction ---
scaffold_ext = ScaffoldExtractor()
scaffold_result = scaffold_ext.extract(mol, mol)  # compare to itself as demo
print(f"Scaffold similarity: {scaffold_result.scaffold_similarity}")

print("\nAll pipeline steps ran successfully!")

Parse status: OK
Fingerprint bits: 2048, on-bits: 57
Drug-likeness: clean=True, violations=2
Scaffold similarity: 1.0

All pipeline steps ran successfully!


In [5]:
# --- A8: ScaffoldExtractor — spec method tests ---
import os, sys

# Test.ipynb sits inside Compund_Matching_Engine/, so the project root is the parent dir.
HERE        = os.getcwd()
PROJECT_ROOT = HERE if os.path.basename(HERE) != "Compund_Matching_Engine" else os.path.dirname(HERE)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("cwd          :", HERE)
print("project root :", PROJECT_ROOT)
print("pkg exists?  :", os.path.isdir(os.path.join(PROJECT_ROOT, "Compund_Matching_Engine")))

# Force-reload in case an older version is cached
for m in list(sys.modules):
    if m.startswith("Compund_Matching_Engine"):
        del sys.modules[m]

from Compund_Matching_Engine import ScaffoldExtractor, parse_smiles

ext = ScaffoldExtractor()

# Two distinct molecules: atorvastatin (statin) vs aspirin (unrelated NSAID)
atorva, _  = parse_smiles("CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O")
aspirin, _ = parse_smiles("CC(=O)Oc1ccccc1C(=O)O")

# 1. extract_murcko returns a Mol
sca = ext.extract_murcko(atorva)
print(f"\nextract_murcko -> {type(sca).__name__}, heavy atoms = {sca.GetNumHeavyAtoms()}")

# 2. murcko_smiles returns canonical SMILES
print(f"murcko_smiles(atorva)  = {ext.murcko_smiles(atorva)}")
print(f"murcko_smiles(aspirin) = {ext.murcko_smiles(aspirin)}")

# 3. compare — self-comparison must give shared_pct == 1.0, novel_atoms == 0
self_cmp = ext.compare(atorva, atorva)
print(f"compare(atorva, atorva) = {self_cmp}")
assert self_cmp["shared_pct"] == 1.0
assert self_cmp["novel_atoms"] == 0

# 4. compare — different molecules: consistent atom counts, shared_pct < 1.0
diff_cmp = ext.compare(atorva, aspirin)
print(f"compare(atorva, aspirin) = {diff_cmp}")
q_size = ext.extract_murcko(atorva).GetNumHeavyAtoms()
assert diff_cmp["shared_atoms"] + diff_cmp["novel_atoms"] == q_size
assert 0.0 <= diff_cmp["shared_pct"] < 1.0

# 5. Legacy extract() must agree with compare()
r = ext.extract(atorva, aspirin)
print(
    f"extract().shared_atoms={r.shared_atoms}, "
    f"novel_atoms={r.novel_atoms}, "
    f"shared_pct={r.shared_pct:.3f}, "
    f"scaffold_similarity={r.scaffold_similarity:.3f}"
)
assert r.shared_atoms == diff_cmp["shared_atoms"]

print("\nAll ScaffoldExtractor spec-method tests PASSED")

cwd          : /Users/girishkrishna/Documents/MSSE_Du/277B Machine Learning/Project_Code/Evidence-Aware-Validation-of-AI-Generated-Drug-Candidates-Using-PPI-Networks/Compund_Matching_Engine
project root : /Users/girishkrishna/Documents/MSSE_Du/277B Machine Learning/Project_Code/Evidence-Aware-Validation-of-AI-Generated-Drug-Candidates-Using-PPI-Networks
pkg exists?  : True
AI-generated molecule: OK
Broken SMILES: ERROR: invalid SMILES string
<class 'rdkit.DataStructs.cDataStructs.ExplicitBitVect'>
2048
Bits ON: 14
Numpy array shape: (2048,)
Numpy array type: int8

extract_murcko -> Mol, heavy atoms = 26
murcko_smiles(atorva)  = O=C(Nc1ccccc1)c1c[nH]c(-c2ccccc2)c1-c1ccccc1
murcko_smiles(aspirin) = c1ccccc1
compare(atorva, atorva) = {'shared_atoms': 26, 'novel_atoms': 0, 'shared_pct': 1.0}
compare(atorva, aspirin) = {'shared_atoms': 6, 'novel_atoms': 20, 'shared_pct': 0.23076923076923078}
extract().shared_atoms=6, novel_atoms=20, shared_pct=0.231, scaffold_similarity=0.094

All ScaffoldE

[00:58:41] SMILES Parse Error: syntax error while parsing: CC(C)c1ccc(INVALID!!!)cc1
[00:58:41] SMILES Parse Error: Failed parsing SMILES 'CC(C)c1ccc(INVALID!!!)cc1' for input: 'CC(C)c1ccc(INVALID!!!)cc1'
